# Cleaning up data for modelling

## Extract SMILES vs Tg data from pdf

In [23]:
import sys
from pathlib import Path
import pandas as pd
pd.set_option('display.max_colwidth', 100)  # Limit to 100 characters

In [24]:
# Ensure project root is on path (one level up from 'development')
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Also add the shared directory so that `utils` package can be resolved (matches streamlit_app logic)
SHARED_DIR = PROJECT_ROOT / 'shared'
print(f"Project root: {PROJECT_ROOT}")

Project root: /home/vidit-agrawal/projects/Polymer-Glass-Transition-Temperature-Predictor


In [25]:
from src.pdf_extractor import extract_tables_from_pdf

In [26]:
pdf_path = PROJECT_ROOT / 'data' / 'raw' / '42004_2024_1305_MOESM3_ESM.pdf'

In [27]:
raw_data_path = PROJECT_ROOT / 'data' / 'processed' / 'Tg_vs_SMILES_extracted_from_pdf.csv'

In [28]:
if not Path(raw_data_path).exists():
    raw_df = extract_tables_from_pdf(
        pdf_path=PROJECT_ROOT / 'data' / 'raw' / '42004_2024_1305_MOESM3_ESM.pdf',
    )
    raw_df.to_csv(raw_data_path)
else:
    raw_df = pd.read_csv(raw_data_path)

PDF has 41 pages


100%|██████████| 41/41 [00:05<00:00,  7.43it/s]

Skipping page 40 due to extraction issues. Will copy manually later.
Skipping page 41 due to extraction issues. Will copy manually later.
Total rows extracted directly from pdf: 888
Total leftover rows added manually: 14
Total rows all together: 902
Removing \n from SMILES and name strings


In [29]:
raw_df.info()
raw_df['logTg'] = raw_df['logTg'].astype(float)
raw_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 902 entries, 0 to 901
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   ID             902 non-null    object
 1   logTg          902 non-null    object
 2   Series         902 non-null    object
 3   Ref            902 non-null    object
 4   name_raw       902 non-null    object
 5   SMILES_raw     902 non-null    object
 6   SMI_generator  902 non-null    object
 7   SMILES         902 non-null    object
 8   name           902 non-null    object
dtypes: object(9)
memory usage: 63.6+ KB


,logTg
count,902.000000
mean,2.559033
std,0.142120
min,2.113900
25%,2.472800
50%,2.564200
75%,2.656100
max,2.909100


## Clean up SMILES notations

In [30]:
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
def remove_salt_and_convert_to_cannonical_smiles(smiles):
    """
    """
    # Parse SMILES string into molecule object
    mol = Chem.MolFromSmiles(smiles)
    # Check for invalid SMILES
    if mol is None:
        print(f"Could not parse SMILES: {smiles}")
        return None

    # Remove salts
    mol_no_salt = SaltRemover().StripMol(mol)
    
    # Convert back to canonical SMILES
    canonical_smiles = Chem.MolToSmiles(
        mol, # molecule object
        canonical=True # ensure canonical form
    )
    
    return canonical_smiles

In [31]:
raw_df.loc[0,'SMILES_raw']

'C=CC(=C)Cl'

In [32]:
raw_df.loc[0,'SMILES']

'C=CC(=C)Cl'

In [33]:
raw_df_copy = raw_df.copy()

In [34]:
for i,row in raw_df_copy.iterrows():
    smile_check = remove_salt_and_convert_to_cannonical_smiles(row['SMILES'])
    if smile_check is None:
        print(row['ID'])
        print(row['SMILES'][0], '0')

[22:03:04] Explicit valence for atom # 11 O, 3, is greater than permitted
[22:03:04] SMILES Parse Error: syntax error while parsing: Si(C)(c1ccccc1)CCC
[22:03:04] SMILES Parse Error: check for mistakes around position 2:
[22:03:04] Si(C)(c1ccccc1)CCC
[22:03:04] ~^
[22:03:04] SMILES Parse Error: Failed parsing SMILES 'Si(C)(c1ccccc1)CCC' for input: 'Si(C)(c1ccccc1)CCC'
[22:03:04] SMILES Parse Error: syntax error while parsing: Si(C)(c1ccc(cc1)N(C)C)CCC
[22:03:04] SMILES Parse Error: check for mistakes around position 2:
[22:03:04] Si(C)(c1ccc(cc1)N(C)C)CCC
[22:03:04] ~^
[22:03:04] SMILES Parse Error: Failed parsing SMILES 'Si(C)(c1ccc(cc1)N(C)C)CCC' for input: 'Si(C)(c1ccc(cc1)N(C)C)CCC'
[22:03:04] SMILES Parse Error: syntax error while parsing: Si(C)(C)c1ccc(cc1)Si(C)(C)OSi(c1ccccc1)(c1ccccc1)O
[22:03:04] SMILES Parse Error: check for mistakes around position 2:
[22:03:04] Si(C)(C)c1ccc(cc1)Si(C)(C)OSi(c1ccccc1)(c
[22:03:04] ~^
[22:03:04] SMILES Parse Error: Failed parsing SMILES 'Si(C

Could not parse SMILES: OC(=O)C1=CC(=CC=C1)C(=O)[O]1C2=CC1=CC=C2
BWN_257
O 0
Could not parse SMILES: Si(C)(c1ccccc1)CCC
NS_059
S 0
Could not parse SMILES: Si(C)(c1ccc(cc1)N(C)C)CCC
NS_073
S 0
Could not parse SMILES: Si(C)(C)c1ccc(cc1)Si(C)(C)OSi(c1ccccc1)(c1ccccc1)O
NS_079
S 0
Could not parse SMILES: Si(c1ccc (C)cc1)(c1ccc(C)cc1)CCC
NS_100
S 0
Could not parse SMILES: Si(c1ccccc1)(c1ccc(cc1)N(C)C)CCC
NS_122
S 0


In [35]:
raw_df['SMILES'].str.contains('Si').sum()

16

In [36]:
raw_df[
    raw_df['SMILES'].str.contains('Si')
]['SMILES']

54                                                       CC1=CC=C(C=C1)[Si]O
92                                                                C[SiH](C)O
139                                                        O[SiH](C)c1ccccc1
619                                                  CC(=C)C(=O)O[Si](C)(C)C
669                                                           [SiH](O)(CC)CC
671                                                               C[Si](=C)C
672                                                      C[Si](C)(O)c1ccccc1
673                                                    [SiH](C)(CCC(F)(F)F)O
675                                                           [SiH](C)(C)CCC
678                                                       Si(C)(c1ccccc1)CCC
680                                                Si(C)(c1ccc(cc1)N(C)C)CCC
681                       Si(C)(C)c1ccc(cc1)Si(C)(C)OSi(c1ccccc1)(c1ccccc1)O
682                                         Si(c1ccc (C)cc1)(c1ccc(C)cc1)CCC

In [37]:
ID_to_fix_Si = ['NS_059', 'NS_073', 'NS_079', 'NS_122']

In [38]:
for id_ in ID_to_fix_Si:
    raw_df_copy.loc[raw_df_copy['ID']==id_, 'SMILES'] = raw_df_copy[raw_df_copy['ID']==id_]['SMILES'].replace('Si','[Si]', regex=True)

In [39]:
raw_df_copy.loc[raw_df_copy['ID']=='NS_100', 'SMILES'] = "[Si](c1ccc(C)cc1)(c1ccc(C)cc1)CCC"

In [40]:
raw_df_copy.loc[raw_df_copy['ID']=='BWN_257', 'SMILES'] = "O=C(Oc1cccc(OC(=O)c2cccc(c2)C=O)c1)c3cccc(C=O)c3"

In [41]:
for i,row in raw_df_copy.iterrows():
    smile_check = remove_salt_and_convert_to_cannonical_smiles(row['SMILES'])
    if smile_check is None:
        print(row['ID'])
        print(row['SMILES'][0], '0')

In [42]:
raw_df_copy['SMILES_clean'] = raw_df_copy['SMILES'].map(remove_salt_and_convert_to_cannonical_smiles)

In [43]:
raw_df_copy['SMILES_clean'].isna().sum()

0

In [44]:
cleaned_data_path = PROJECT_ROOT / 'data' / 'processed' / 'Tg_vs_SMILES_all.csv'
raw_df_copy.to_csv(cleaned_data_path)